# LLM Proof Benchmark — Google Colab workflow

Run these cells from top to bottom. Before starting, choose **Runtime → Change runtime type → T4 GPU**. Colab's `/content` folder is temporary, so download the final archive before the runtime ends.

## 1. Clone or update the project

**Outcome:** the repository is available at `/content/llm-proof-benchmark`. Re-running this cell updates an existing copy instead of cloning a second one.

In [ ]:
from pathlib import Path

%cd /content
if not Path('llm-proof-benchmark').exists():
    !git clone https://github.com/WaydenDunford/llm-proof-benchmark.git
%cd /content/llm-proof-benchmark
!git pull --ff-only

## 2. Confirm the GPU

**Outcome:** the output should name a Tesla T4 and say `CUDA available: True`. If it does not, change the runtime type to T4 GPU and run this cell again.

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 3. Install benchmark dependencies

**Outcome:** the project is installed in editable mode, while Colab's GPU-enabled PyTorch remains in place. `bitsandbytes` enables 4-bit loading so the 7B model fits on the T4.

In [ ]:
%cd /content/llm-proof-benchmark
!pip install -q -e '.[dev]' transformers accelerate bitsandbytes

## 4a. DeepSeek-R1-Distill-Qwen-7B

**Outcome:** only DeepSeek-R1-Distill-Qwen-7B is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'deepseek-r1-distill-qwen-7b'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4b. DeepTheorem-Qwen-7B-RL

**Outcome:** only DeepTheorem-Qwen-7B-RL is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'deeptheorem-qwen-7b'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4c. Qwen3-8B

**Outcome:** only Qwen3-8B is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'qwen3'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4d. All three models, sequentially

**Outcome:** all models are selected for one comparison run. They are not loaded together: the benchmark loads one 4-bit model, generates a proof, unloads it, clears GPU memory, and then loads the next. Including Math-Shepherd, the first all-model run downloads roughly 55–60 GB to temporary Colab storage and can take much longer. Run this cell, then run cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
for model in config['models']:
    model['enabled'] = True
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Sequential comparison:', [m['name'] for m in config['models'] if m['enabled']])

## 5. Generate a real proof and create the evaluation files

**Outcome:** a real model-generated proof, Math-Shepherd step scores, canonical proofs/evaluations JSON files, an Excel workbook, and a blind Markdown bundle are created. For one selected model, the first run downloads roughly 30 GB of model files into the temporary Colab runtime: the generator plus the separate Math-Shepherd checkpoint. An all-model sequential run needs roughly 55–60 GB and may take substantially longer.

Math-Shepherd loads only after the generator has been unloaded. It uses 4-bit loading on the T4, so its scores are real process-reward scores rather than mock plumbing data.

In [ ]:
%cd /content/llm-proof-benchmark
!python -m src.cli prepare-dual-evaluation --theorem T001 --prompt structured --max-new-tokens 512 --temperature 0 --no-sample --math-shepherd-backend transformers

## 6. Inspect the latest proof without flooding the notebook output

**Outcome:** you see the latest run ID, its model status, and an 800-character proof preview. The full proof remains in the JSON file.

In [ ]:
import json
from pathlib import Path

proofs_file = sorted(Path('results/proofs').glob('T001_*_proofs.json'))[-1]
proofs = json.loads(proofs_file.read_text(encoding='utf-8'))
print('Proofs file:', proofs_file)
print('Run ID:', proofs['run_id'])
for proof in proofs['proofs']:
    print(f"\n{proof['model_name']}: {proof['status']}")
    print(proof.get('proof_text', '')[:800])

## 7. Download the anonymous ChatGPT evaluation bundle

**Outcome:** your browser downloads one Markdown file. Upload this file to ChatGPT. Do not upload files from `results/evaluation_maps`, because they identify the model behind each anonymous proof.

In [ ]:
from google.colab import files
from pathlib import Path

bundle_file = sorted(Path('results/evaluation_input').glob('T001_*_chatgpt_bundle.md'))[-1]
print('Downloading:', bundle_file)
files.download(str(bundle_file))

## 8. Evaluate the bundle in ChatGPT

Upload the downloaded Markdown bundle to ChatGPT and follow its instruction to return **only the required JSON**. Save that JSON response as a file on your computer. Keep the proof IDs unchanged.

## 9. Upload and import ChatGPT's JSON evaluation

**Outcome:** the JSON evaluation is validated and merged into the same canonical evaluations file. The Excel workbook and reports are updated. Run this only after completing the ChatGPT evaluation.

In [ ]:
from google.colab import files
import json
from pathlib import Path

uploaded = files.upload()
evaluation_file = Path(next(iter(uploaded)))
evaluation = json.loads(evaluation_file.read_text(encoding='utf-8-sig'))
run_id = evaluation['run_id']
print('Importing evaluation for run:', run_id)
!python -m src.cli import-chatgpt-evaluation --run-id {run_id} --evaluation {evaluation_file}

## 10. Download the finished results

**Outcome:** your browser downloads a zip archive containing the run's proofs, evaluations, reports, and Excel workbook. This keeps the results after Colab clears `/content`.

In [ ]:
from google.colab import files

!zip -r /content/llm-proof-benchmark-results.zip results/proofs results/evaluations results/reports results/benchmark_results.xlsx
files.download('/content/llm-proof-benchmark-results.zip')

## Optional: copy results to Google Drive

**Outcome:** the finished zip is copied to `MyDrive/llm-proof-benchmark`. Run this after the download cell if you also want a Drive backup.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
drive_folder = Path('/content/drive/MyDrive/llm-proof-benchmark')
drive_folder.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/llm-proof-benchmark-results.zip', drive_folder / 'llm-proof-benchmark-results.zip')
print('Saved:', drive_folder / 'llm-proof-benchmark-results.zip')